CA1 – Human Centered Deep Learning

Name: Saketh Reddy Chandupatla

Student Number: x00231080

## Section 6: Grid Search 

To better optimize our neural network's performance, a comprehensive grid search is conducted using KerasClassifier with GridSearchCV from Scikit-learn. My aim was to systematically evaluate the impact of various batch sizes, optimizers and epoch values while leveraging Henormal weight initialization and early stopping to prevent overfitting.

In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.12.0


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier

In [3]:
# 1. Load and preprocess data
# -----------------------------
# Loading the dataset
df = pd.read_csv("X00231080.csv")

# Split features and target
X = df.drop("Diabetes_binary", axis=1)
y = df["Diabetes_binary"]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split with stratification
seed = 1
np.random.seed(seed)
tf.random.set_seed(seed)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.33, random_state=seed, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (28418, 21)
Test set size: (13997, 21)


**6.1 Experimental Setup**

* Model Architecture: 32-16-8 dense layers with ReLU activation, HeNormal initialization, and a sigmoid output layer.
* Dataset Split: 67% training, 33% test with stratified sampling.
* Regularization & Stability:
* EarlyStopping (patience=10, monitored 'loss') to halt training when improvement plateaued.
* ModelCheckpoint to store the best-performing model (based on accuracy).
* Hyperparameter Grid:
  - Batch Sizes: 16, 32, 128, 256
  - Epochs: 30, 50, 100
  - Optimizers: Adam, RMSprop

A total of 24 combinations were evaluated using 3-fold cross-validation on the training set.

In [4]:
# 2. Build model function
# -----------------------------
def build_model(optimizer='adam'):
    model = Sequential()
    model.add(Dense(32, activation='relu', input_dim=X_train.shape[1], kernel_initializer=HeNormal()))
    model.add(Dense(16, activation='relu', kernel_initializer=HeNormal()))
    model.add(Dense(8, activation='relu', kernel_initializer=HeNormal()))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [5]:
# 3. Wrap with KerasClassifier
# -----------------------------
model = KerasClassifier(build_fn=build_model, verbose=0)

C:\Users\suzak\AppData\Local\Temp\ipykernel_15344\1504816404.py:3: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  model = KerasClassifier(build_fn=build_model, verbose=0)


In [6]:
# 4. Set hyperparameter grid
# -----------------------------
param_grid = {
    'batch_size': [16, 32, 128, 256],
    'epochs': [30, 50, 100],
    'optimizer': ['adam', 'rmsprop']
}

In [7]:
# 5. Define callbacks
# -----------------------------
early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
model_checkpoint = ModelCheckpoint("best_model.h5", monitor='accuracy', save_best_only=True, mode='max', verbose=1)
callbacks_list = [early_stop, model_checkpoint]

In [8]:
# 6. Run Grid Search
# -----------------------------
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, verbose=2)
grid_result = grid.fit(X_train, y_train, callbacks=callbacks_list)

Fitting 3 folds for each of 24 candidates, totalling 72 fits

Epoch 1: accuracy improved from -inf to 0.70947, saving model to best_model.h5

Epoch 2: accuracy improved from 0.70947 to 0.74294, saving model to best_model.h5

Epoch 3: accuracy improved from 0.74294 to 0.74706, saving model to best_model.h5

Epoch 4: accuracy improved from 0.74706 to 0.75044, saving model to best_model.h5

Epoch 5: accuracy improved from 0.75044 to 0.75070, saving model to best_model.h5

Epoch 6: accuracy improved from 0.75070 to 0.75260, saving model to best_model.h5

Epoch 7: accuracy improved from 0.75260 to 0.75366, saving model to best_model.h5

Epoch 8: accuracy improved from 0.75366 to 0.75571, saving model to best_model.h5

Epoch 9: accuracy improved from 0.75571 to 0.75629, saving model to best_model.h5

Epoch 10: accuracy improved from 0.75629 to 0.75809, saving model to best_model.h5

Epoch 11: accuracy improved from 0.75809 to 0.75888, saving model to best_model.h5

Epoch 12: accuracy did not

In [9]:
# 7. Print results
# -----------------------------
print(f"\n Best Accuracy: {grid_result.best_score_:.4f}")
print(f" Best Parameters: {grid_result.best_params_}")

print("\n All Grid Search Results:")
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']

for mean, std, param in zip(means, stds, params):
    print(f"{mean:.4f} (+/- {std:.4f}) with: {param}")


 Best Accuracy: 0.7484
 Best Parameters: {'batch_size': 256, 'epochs': 30, 'optimizer': 'adam'}

 All Grid Search Results:
0.7369 (+/- 0.0026) with: {'batch_size': 16, 'epochs': 30, 'optimizer': 'adam'}
0.7435 (+/- 0.0031) with: {'batch_size': 16, 'epochs': 30, 'optimizer': 'rmsprop'}
0.7345 (+/- 0.0066) with: {'batch_size': 16, 'epochs': 50, 'optimizer': 'adam'}
0.7447 (+/- 0.0055) with: {'batch_size': 16, 'epochs': 50, 'optimizer': 'rmsprop'}
0.7273 (+/- 0.0022) with: {'batch_size': 16, 'epochs': 100, 'optimizer': 'adam'}
0.7364 (+/- 0.0040) with: {'batch_size': 16, 'epochs': 100, 'optimizer': 'rmsprop'}
0.7400 (+/- 0.0070) with: {'batch_size': 32, 'epochs': 30, 'optimizer': 'adam'}
0.7411 (+/- 0.0036) with: {'batch_size': 32, 'epochs': 30, 'optimizer': 'rmsprop'}
0.7365 (+/- 0.0068) with: {'batch_size': 32, 'epochs': 50, 'optimizer': 'adam'}
0.7370 (+/- 0.0038) with: {'batch_size': 32, 'epochs': 50, 'optimizer': 'rmsprop'}
0.7332 (+/- 0.0027) with: {'batch_size': 32, 'epochs': 100,

**6.2 Grid search Results summary**

| **Batch Size** | **Epochs** | **Optimizer** | **Accuracy (Mean ± Std)**      |
| -------------- | ---------- | ------------- | ------------------------------ |
| 256            | 30         | Adam          | **0.7484 ± 0.0034**   *(Best)* |
| 128            | 30         | RMSprop       | 0.7471 ± 0.0019                |
| 256            | 50         | RMSprop       | 0.7469 ± 0.0040                |
| 256            | 30         | RMSprop       | 0.7466 ± 0.0052                |
| 128            | 30         | Adam          | 0.7429 ± 0.0025                |
| 16             | 50         | RMSprop       | 0.7447 ± 0.0055                |
| 16             | 30         | RMSprop       | 0.7435 ± 0.0031                |
| 32             | 30         | RMSprop       | 0.7411 ± 0.0036                |



The best performing configuration was: Batch Size = 256, Epochs = 30, Optimizer = Adam
Achieved Accuracy: 0.7484


**6.3 Performance Insights**

1. Batch size: Larger batch sizes(128 and 256) consistently outperformed smaller batch sizes. mainly batch size 256 which yielded the best performance, likely due to more stable gradient updates and efficient hardware utilization.
2. Epochs: Peak accuracy was usually achieved within 30 to 50 epochs beyond which performance was likely plateaued or slightly dropped. This confirms that early stopping was effective in reducing overtraining risk.
3. Optimizer comparison: Adam had slight edge in stability and generalization. Whereas RMSprop performed closely especially with smaller batch sizes but was marginally less consistent across epochs
4. Weight Initialization: Using henormal contributed to faster convergence and better handling of ReLU activations therefore supporting the deeper architecture's stability.

After performing Grid search I observed a clear trend : 
Adam with batch size 256 and early stopping at roughly 30 epochs consistently gives the highest validation accuracy.

The model benefits from HeNormal initializer and both Adam and RMSprop are effective optimizers depending on batch size.

This hyperparameter tuning phase played a critical role in identifying a configuration that balances accuracy, stability and training efficiency. The selected best model will be used in further evaluation and deployment stages.